<center><font size=8>LLM-Based Medical Assistant
</center></font>

<center><p float="center">
  <img src="https://www.champsoft.com/wp-content/uploads/2025/09/Large-Language-Model-Healthcare-Software.webp" width=800></a>

### **Business Context**

Accessing medical information can be difficult for patients and healthcare professionals because relevant information is often spread across lengthy medical documents. A Medical Assistant powered by an LLM can make this process easier by allowing users to ask questions in natural language and receive quick, understandable responses. It can help with medical explanations, information retrieval, and summarization while providing suitable guidance. The system is designed as an informational support tool and does not replace doctors or professional medical advice.




### **Objective**

The main objective is to build an LLM-powered Medical Assistant that can interact with users, understand their health-related queries, and provide informative responses in a simple and conversational manner.

####The system aims to:

- Provide answers to common health and medical queries.
- Simplify complex medical terms and concepts.
- Help users understand general medical information.
- Generate concise summaries from medical content.
- Use previous conversation context to provide relevant responses.
- Use an LLM to produce natural and meaningful answers.
- Include safety recommendations and suggest professional medical consultation when appropriate.


### **Goal**

To develop an LLM-based Medical Assistant that provides simple, clear, and relevant answers to general medical questions and helps users understand health-related information easily.

### **Technology**

**LLM only** — no RAG, vector database, embeddings, or external knowledge-retrieval system.

**Flow:**

`User Query → LLM → Medical Response`


## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Install required libraries
!pip install -q \
langchain==0.3.27 \
langchain-community==0.3.27 \
chromadb==1.0.15 \
pymupdf==1.26.3 \
tiktoken==0.9.0 \
datasets==4.0.0 \
evaluate==0.4.5 \
langchain-openai==0.3.30 \
langchain-huggingface \
sentence-transformers \
huggingface-hub \
transformers

In [ ]:
# Install llama-cpp-python with GPU support for running LLaMA models
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.45 --force-reinstall --upgrade --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 331.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 304.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 331.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 296.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 306.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
langgraph 1.2.9 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.4

In [ ]:
# Install Hugging Face Hub client library for downloading models
!pip install huggingface_hub==0.20.3 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 0.3.1 requires huggingface-hub>=0.33.4, but you have huggingface-hub 0.20.3 which is incompatible.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.20.3 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.20.3 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.20.3 which is incompatible.
sentence-transformers 5.6.0 requires huggingface-hub>=0.23.0, but you have huggingface-hub 0.20.3 which is incompatible.
transformers 5.13.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.20.3 which is incompatible.
accelerate 1.14.0 requires huggingface_hub>=0.21.0, but you have huggingface-hub 0.20.3 which is incompatible.
gradio 6.20.0 requires hugg

In [ ]:
!pip install sentence-transformers

In [ ]:
import pandas as pd
import os

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

## **LLM with Prompt Engineering Response**

#### **Download LLaMA-2 13B Chat Model**

In [ ]:
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf"

model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

print(model_path)

/root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGUF/snapshots/4458acc949de0a9914c3eab623904d4fe999050a/llama-2-13b-chat.Q5_K_M.gguf


#### **Initialize LLaMA Model with Configuration**

In [ ]:
lcpp_llm = Llama(
    model_path=model_path, # Path to the downloaded GGUF model
    n_threads=4,           # Number of CPU threads to use
    n_batch=512,           # Batch size for prompt processing
    n_gpu_layers=-1,       # Number of layers to offload to GPU (-1 for all)
    n_ctx=4096             # Context window
)

llama_model_loader: loaded meta data with 19 key-value pairs and 363 tensors from /root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGUF/snapshots/4458acc949de0a9914c3eab623904d4fe999050a/llama-2-13b-chat.Q5_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 5120
llama_model_loader: - kv   4:                          llama.block_count u32              = 40
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 13824
llama_model_loader: - kv   6:                 llama.rope.dimension_

In [ ]:
# Provides an example and answer anchor to guide the model in giving concise, evidence-based responses without echoing the question.
system_prompt = """
You are a medical expert AI assistant. Respond with concise, evidence-based answers.
"""

user_prompt = """
Example:
Q: What are the common symptoms of appendicitis?
A: Common symptoms include abdominal pain (usually starting near the navel), nausea, vomiting, and fever.
References: Mayo Clinic, UpToDate

Now answer:
Q: What is the protocol for managing sepsis in a critical care unit?
A:
"""


#### **Response Function**

In [ ]:
# function to generate, process, and return the response from the LLM
def prompt_engineering_response(user_prompt):
    # Put the system message first, then the user question, then anchor with "Answer:"
    prompt = f"""{system_prompt}

Question: {user_prompt}

Answer:"""

    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt,
        max_tokens=512,
        temperature=0.2,
        top_p=0.95,
        stop=["Question:", "</s>"]
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"].strip()
    return response_text


## Question Answering using LLM with Prompt Engineering

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"

response_1 = prompt_engineering_response(question_1)

print(response_1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =     276.29 ms /   445 runs   (    0.62 ms per token,  1610.63 tokens per second)
llama_print_timings: prompt eval time =     677.74 ms /    19 tokens (   35.67 ms per token,    28.03 tokens per second)
llama_print_timings:        eval time =   43128.53 ms /   444 runs   (   97.14 ms per token,    10.29 tokens per second)
llama_print_timings:       total time =   45837.01 ms /   463 tokens


The Surviving Sepsis Campaign (SSC) guidelines provide a comprehensive protocol for managing sepsis in a critical care unit. Here are the key recommendations:

1. Early recognition and diagnosis: Use clinical criteria and biomarkers to identify patients with suspected sepsis. Confirm the diagnosis with blood cultures and other tests as needed.
2. Rapid administration of antibiotics: Start broad-spectrum antibiotics within the first hour of recognition of sepsis, and within 3 hours of recognition in severe sepsis or septic shock.
3. Fluid resuscitation: Administer crystalloid fluids initially, and consider vasopressors if fluid resuscitation fails to improve mean arterial pressure (MAP) or if there are signs of hypoperfusion.
4. Oxygenation: Use invasive or non-invasive ventilation as needed to maintain PaO2/FiO2 ratio ≥100 mmHg.
5. Pain management: Use sedation and analgesia to manage pain and distress.
6. Monitoring: Closely monitor vital signs, including MAP, heart rate, and respirat

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

response_2 = prompt_engineering_response(question_2)

print(response_2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =     137.72 ms /   223 runs   (    0.62 ms per token,  1619.29 tokens per second)
llama_print_timings: prompt eval time =     489.15 ms /    61 tokens (    8.02 ms per token,   124.71 tokens per second)
llama_print_timings:        eval time =   21290.04 ms /   222 runs   (   95.90 ms per token,    10.43 tokens per second)
llama_print_timings:       total time =   22787.76 ms /   283 tokens


Appendicitis is inflammation of the appendix that can cause severe pain, nausea, vomiting, fever, and loss of appetite. While antibiotics may be used to treat a mild case of appendicitis, surgery is usually necessary to remove the inflamed appendix. The surgical procedure most commonly used is a laparoscopic appendectomy, where a small incision is made in the abdomen and a laparoscope (a thin tube with a camera and light) is inserted to visualize the appendix. The inflamed appendix is then removed through the small incision. In severe cases, an open appendectomy may be necessary, where a larger incision is made in the abdomen to allow for better visualization of the appendix. It is important to seek medical attention immediately if symptoms of appendicitis are present, as delaying treatment can lead to complications such as the appendix rupturing and spreading infection throughout the abdominal cavity.


### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
question_3 = "What are the effective treatments for sudden patchy hair loss and what are the possible causes?"

response_3 = prompt_engineering_response(question_3)

print(response_3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =     306.49 ms /   512 runs   (    0.60 ms per token,  1670.51 tokens per second)
llama_print_timings: prompt eval time =     720.64 ms /    22 tokens (   32.76 ms per token,    30.53 tokens per second)
llama_print_timings:        eval time =   48202.75 ms /   511 runs   (   94.33 ms per token,    10.60 tokens per second)
llama_print_timings:       total time =   51287.50 ms /   533 tokens


Sudden patchy hair loss can be caused by various factors, including hormonal imbalances, autoimmune disorders, infections, and nutritional deficiencies. Here are some effective treatments for sudden patchy hair loss:

1. Medications: Minoxidil (Rogaine) and finasteride (Propecia) are two medications that have been approved by the FDA for treating hair loss. Minoxidil is applied topically to the scalp and can help stimulate hair growth and slow down hair loss. Finasteride is an oral medication that works by blocking the production of dihydrotestosterone (DHT), a hormone that contributes to hair loss.
2. Low-level laser therapy (LLLT): LLLT uses low-level lasers or light-emitting diodes to stimulate hair growth. It has been shown to increase hair density and promote hair growth in people with androgenetic alopecia (male/female pattern baldness).
3. Platelet-rich plasma (PRP) therapy: PRP therapy involves injecting platelet-rich plasma (PRP) into the scalp to stimulate hair growth. PRP is

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
question_4 = "What treatments are recommended for traumatic brain injury?"

response_4 = prompt_engineering_response(question_4)

print(response_4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =     280.33 ms /   476 runs   (    0.59 ms per token,  1698.02 tokens per second)
llama_print_timings: prompt eval time =     514.25 ms /    15 tokens (   34.28 ms per token,    29.17 tokens per second)
llama_print_timings:        eval time =   45675.53 ms /   475 runs   (   96.16 ms per token,    10.40 tokens per second)
llama_print_timings:       total time =   48273.22 ms /   490 tokens


Traumatic brain injury (TBI) is a complex condition that requires individualized treatment based on the severity and location of the injury. Here are some common treatments recommended for TBI:

1. Medications: Pain management medications, such as analgesics and sedatives, may be prescribed to help manage symptoms such as headaches and agitation. Anti-seizure medications may be used to prevent seizures in patients with seizure disorders.
2. Rehabilitation therapy: Physical, occupational, and speech therapy may be necessary to help patients regain lost functions and improve cognitive and physical abilities.
3. Surgery: In some cases, surgery may be necessary to relieve pressure on the brain or repair damaged blood vessels.
4. Neurointensive care: Patients with severe TBI may require neurointensive care, which includes close monitoring and supportive care in an intensive care unit.
5. Rehabilitation programs: Patients may benefit from rehabilitation programs that focus on improving cogni

#### **Create and Display Results DataFrame**

In [ ]:
# Create the DataFrame

prompt_result_df = pd.DataFrame({
    "questions": [question_1, question_2, question_3, question_4],
    "prompt_Engineering_responses": [response_1 ,response_2 ,response_3 ,response_4 ]})

# Display the DataFrame
prompt_result_df.head()

,questions,prompt_Engineering_responses
0,What is the protocol for managing sepsis in a ...,The Surviving Sepsis Campaign (SSC) guidelines...
1,"What are the common symptoms for appendicitis,...",Appendicitis is inflammation of the appendix t...
2,What are the effective treatments for sudden p...,Sudden patchy hair loss can be caused by vario...
3,What treatments are recommended for traumatic ...,Traumatic brain injury (TBI) is a complex cond...


### **Observations**

* The LLaMA model successfully generated responses for the medical queries using only the provided prompts and its pretrained knowledge.
* Prompt engineering helped produce concise and well-structured responses without using external medical documents.
* The quality and relevance of the responses depended strongly on the clarity of the system and user prompts.
* Since no external knowledge source was used, the responses were not always grounded in authoritative medical references and may contain unsupported information or hallucinations.



### **Loading the data**

In [ ]:
# Mount Google Drive to the /content/drive directory to access the files
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

# Load Merck Manual PDF
pdf_path = "/content/drive/MyDrive/p/medical_diagnosis_manual.pdf"
loader = PyMuPDFLoader(pdf_path)

# Load as LangChain Documents
document = loader.load()

print(f"Total pages loaded: {len(document)}")



Total pages loaded: 4114


**There are `4114` pages in the document pdf and pdf loaded successfully**

### Data Overview

Display the content of page number 16 and 17

In [ ]:
for i in range(15, 17):
    print("=" * 100)
    print(f"PAGE {i+1}")
    print("=" * 100)

    print(document[i].page_content[:3000])

    print("\n")

PAGE 16
degree. The book received critical acclaim and sold over 2 million copies. The Second Home Edition was
released in 2003. Merck's commitment to providing comprehensive, understandable medical information
to all people continued with The Merck Manual Home Health Handbook, published in 2009.
The Merck Manual of Health & Aging , published in 2004, continued Merck's commitment to education
and geriatric care, providing information on aging and the care of older people in words understandable
by the lay public.
In 2008, The Merck Manual of Patient Symptoms  was introduced to complement The Merck Manual
and was intended to help newcomers to clinical diagnosis approach patients who present with certain
common symptoms.
As part of its commitment to ensuring that all who need and want medical information can get it, Merck
provides the content of these Merck Manuals on the web for free (www.merckmanuals.com). Registration
is not required, and use is unlimited. The web publications are con

## **Data Chunking**

Split the document into Chunks and display the total chunks

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(document)

print(f"Total chunks: {len(docs)}")


Total chunks: 17984


### Embedding

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Creating a Vector Database

In [ ]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=out_dir)

In [ ]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

### Retriever

Retrieval and Response Generation using Vector Search

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [ ]:
medical_system_message = """
You are an AI assistant designed to support healthcare professionals by providing evidence-based, concise, and accurate responses using authoritative medical sources, such as the Merck Manuals.

Your goal is to help clinicians, researchers, and healthcare teams quickly access reliable medical knowledge to improve patient outcomes, support decision-making, and reduce information overload.

User input will include context extracted from trusted medical sources. This context will begin with the token:

###Context
The context may include excerpts from the Merck Manuals, clinical guidelines, or peer-reviewed medical literature, including titles, sections, authors, and other relevant metadata.

When crafting your response:
- Use only the provided context to answer the question.
- Provide concise, clinically relevant, and accurate answers.
- Include the source (title, section, and page/section reference) when applicable.
- If the context does not contain relevant information, respond: "Sorry, this is out of my knowledge base."
- Do NOT provide personal medical advice or treatment recommendations outside of the context.
- Maintain a professional, neutral, and safe tone appropriate for healthcare communication.

Example response format:

Answer:
[Answer based on context]

Source:
[Source title, section, page]
"""


In [ ]:
medical_user_message_template = """
###Context
Here are relevant excerpts from the Merck Manuals or other authoritative medical sources:
{context}

###Question
{question}
"""


### Response Function

In [ ]:
# Function to retrieve relevant context and generate a RAG response
def generate_rag_response(user_input, retriever,
                          system_message, user_message_template,
                          k=5, max_tokens=500,
                          temperature=0.3, top_p=0.95):

    # Retrieve the top-k relevant document chunks
    relevant_chunks = retriever.invoke(user_input)

    if not relevant_chunks:
        return "Sorry, this is out of my knowledge base."

    # Combine retrieved chunks with source information
    context_for_query = "\n\n".join(
        [
            f"Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}"
            for doc in relevant_chunks
        ]
    )

    # Build the user prompt
    user_message = user_message_template.format(
        context=context_for_query,
        question=user_input
    )

    # Combine system prompt and user prompt
    prompt = f"""{system_message}

{user_message}

Answer:
"""

    # Generate the response using the local LLaMA model
    try:
        response = lcpp_llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            stop=["###Question", "###Context"],
            echo=False
        )

        return response["choices"][0]["text"].strip()

    except Exception as e:
        return f"Sorry, I encountered the following error:\n{e}"

## Output Evaluation

In [ ]:
medical_groundedness_rater_system_message = """
You are tasked with rating AI-generated answers to medical questions posed by healthcare professionals.
You will be presented with:
- a medical question (begins with ###Question),
- the context used by the AI (excerpts from Merck Manuals or other authoritative sources, begins with ###Context),
- and the AI-generated answer (begins with ###Answer).

Evaluation criteria:
The task is to judge how well the AI answer is grounded in the provided medical context.

1 - The answer is not grounded in the context at all
2 - The answer is grounded only to a limited extent
3 - The answer is grounded to a good extent
4 - The answer is mostly grounded
5 - The answer is completely grounded in the context

Instructions:
1. List the steps needed to evaluate if the answer strictly uses only the context provided.
2. Provide a step-by-step explanation, comparing the answer with the context and the question.
3. Assign a groundedness score based on the above evaluation.
4. Return only the final score in dictionary format (not JSON), e.g.: {groundedness_score:4}
Score should be in the range 1 to 5.
"""


In [ ]:
medical_relevance_rater_system_message = """
You are tasked with rating AI-generated answers to medical questions posed by healthcare professionals.
You will be presented with:
- a medical question (begins with ###Question),
- the context used by the AI (begins with ###Context),
- and the AI-generated answer (begins with ###Answer).

Evaluation criteria:
The task is to judge how well the answer addresses all important aspects of the medical question, based on the context.

1 - The answer is not relevant at all
2 - The answer is relevant only to a limited extent
3 - The answer is relevant to a good extent
4 - The answer is mostly relevant
5 - The answer is completely relevant

Instructions:
1. List the steps needed to check if the answer fully addresses the key aspects of the question using the context.
2. Provide a step-by-step explanation evaluating the relevance.
3. Assign a relevance score based on the evaluation.
4. Return only the final score in dictionary format (not JSON), e.g.: {relevance_score:4}
Score should be in the range 1 to 5.
"""


In [ ]:
medical_rater_user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""


In [ ]:
# Function to evaluate Groundedness and Relevance of a RAG response
def generate_ground_relevance_response(user_input, response, retriever,
                                       groundedness_system_message,
                                       relevance_system_message,
                                       user_message_template,
                                       k=5, max_tokens=500,
                                       temperature=0, top_p=0.95):

    # Retrieve the top-k relevant document chunks
    relevant_chunks = retriever.invoke(user_input)

    if not relevant_chunks:
        return "No context found.", "No context found."

    # Combine retrieved chunks into a single context string
    context = "\n\n".join(
        [
            f"Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}"
            for doc in relevant_chunks
        ]
    )

    # Build the evaluation prompt
    user_message = user_message_template.format(
        question=user_input,
        context=context,
        answer=response
    )

    # ---------------- Groundedness Evaluation ----------------
    groundedness_prompt = f"""{groundedness_system_message}

{user_message}

Score:
"""

    groundedness_response = lcpp_llm(
        prompt=groundedness_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        echo=False
    )

    groundedness_result = groundedness_response["choices"][0]["text"].strip()


    relevance_prompt = f"""{relevance_system_message}

{user_message}

Score:
"""

    relevance_response = lcpp_llm(
        prompt=relevance_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        echo=False
    )

    relevance_result = relevance_response["choices"][0]["text"].strip()

    # Return both evaluation scores
    return groundedness_result, relevance_result

#### Evaluation

In [ ]:
llm_judge_prompt_ground_1, llm_judge_prompt_rel_1 = generate_ground_relevance_response(
    question_1,
    response_1,
    retriever,

    groundedness_system_message=medical_groundedness_rater_system_message,

    relevance_system_message=medical_relevance_rater_system_message,

    user_message_template=medical_rater_user_message_template

)

# Print the results
print(llm_judge_prompt_ground_1, end="\n\n")
print(llm_judge_prompt_rel_1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =      59.20 ms /    96 runs   (    0.62 ms per token,  1621.59 tokens per second)
llama_print_timings: prompt eval time =    6452.91 ms /  2149 tokens (    3.00 ms per token,   333.03 tokens per second)
llama_print_timings:        eval time =   10307.27 ms /    95 runs   (  108.50 ms per token,     9.22 tokens per second)
llama_print_timings:       total time =   17272.44 ms /  2244 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =     102.93 ms /   192 runs   (    0.54 ms per token,  1865.29 tokens per second)
llama_print_timings: prompt eval time =    7584.86 ms /  2077 tokens (    3.65 ms per token,   273.83 tokens per second)
llama_print_timings:        eval time =   21645.77 ms /   191 runs   (  113.33 ms per token,     8.82 tokens per second)
llama_print_timings:       to

The answer is completely grounded in the provided medical context. The Surviving Sepsis Campaign guidelines are widely recognized and accepted as the standard of care for sepsis management, and the answer provides a comprehensive summary of these guidelines. The answer is well-structured and easy to follow, and it includes specific recommendations for managing sepsis in a critical care unit. Therefore, the groundedness score is 5.

Relevance Score: 4

Explanation:

The answer provides a comprehensive protocol for managing sepsis in a critical care unit, based on the SSC guidelines. The answer covers all key aspects of sepsis management, including early recognition and diagnosis, antibiotic administration, fluid resuscitation, oxygenation, pain management, monitoring, source control, nutritional support, and hemodialysis or hemofiltration. The answer also emphasizes the importance of a multidisciplinary team approach in managing sepsis patients. However, the answer does not provide spec

In [ ]:
llm_judge_prompt_ground_2, llm_judge_prompt_rel_2 = generate_ground_relevance_response(

    question_2,
    response_2,
    retriever,

    groundedness_system_message=medical_groundedness_rater_system_message,

    relevance_system_message=medical_relevance_rater_system_message,

    user_message_template=medical_rater_user_message_template

)

# Print the results
print(llm_judge_prompt_ground_2, end="\n\n")
print(llm_judge_prompt_rel_2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =      76.54 ms /    97 runs   (    0.79 ms per token,  1267.34 tokens per second)
llama_print_timings: prompt eval time =    5538.94 ms /  1967 tokens (    2.82 ms per token,   355.12 tokens per second)
llama_print_timings:        eval time =    9180.94 ms /    96 runs   (   95.63 ms per token,    10.46 tokens per second)
llama_print_timings:       total time =   15383.88 ms /  2063 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =      54.31 ms /    93 runs   (    0.58 ms per token,  1712.23 tokens per second)
llama_print_timings: prompt eval time =    5635.32 ms /  1944 tokens (    2.90 ms per token,   344.97 tokens per second)
llama_print_timings:        eval time =    9595.52 ms /    92 runs   (  104.30 ms per token,     9.59 tokens per second)
llama_print_timings:       to

The answer is not grounded in the context at all. The answer does not mention any of the specific symptoms, signs, or etiology of appendicitis described in the provided medical context. The answer is not based on any of the information provided in the context and does not provide any specific treatment recommendations based on the severity of the condition. Therefore, the answer is not grounded in the context at all and scores a 1 out of 5.

The answer is completely relevant, addressing all important aspects of the medical question. The answer provides a clear and concise overview of appendicitis, including its symptoms, diagnosis, and treatment options. The answer also highlights the importance of seeking medical attention immediately if symptoms of appendicitis are present, which is crucial for effective management of the condition. Therefore, I assign a relevance score of 5 out of 5.


In [ ]:
llm_judge_prompt_ground_3, llm_judge_prompt_rel_3 = generate_ground_relevance_response(

    question_3,
    response_3,
    retriever,

    groundedness_system_message=medical_groundedness_rater_system_message,

    relevance_system_message=medical_relevance_rater_system_message,

    user_message_template=medical_rater_user_message_template

)

# Print the results
print(llm_judge_prompt_ground_3, end="\n\n")
print(llm_judge_prompt_rel_3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =      73.04 ms /   127 runs   (    0.58 ms per token,  1738.73 tokens per second)
llama_print_timings: prompt eval time =    7004.46 ms /  2286 tokens (    3.06 ms per token,   326.36 tokens per second)
llama_print_timings:        eval time =   13958.16 ms /   126 runs   (  110.78 ms per token,     9.03 tokens per second)
llama_print_timings:       total time =   21570.76 ms /  2412 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =      52.49 ms /    91 runs   (    0.58 ms per token,  1733.66 tokens per second)
llama_print_timings: prompt eval time =    7103.78 ms /  2263 tokens (    3.14 ms per token,   318.56 tokens per second)
llama_print_timings:        eval time =    9701.76 ms /    90 runs   (  107.80 ms per token,     9.28 tokens per second)
llama_print_timings:       to

The answer is not completely grounded in the provided medical context. While it mentions some possible causes of sudden patchy hair loss, it does not provide any specific information about the causes of hair loss associated with hyperandrogenemia, traction alopecia, tinea capitis, or trichotillomania, which are all mentioned in the provided medical context. Additionally, the answer does not provide any information about the evaluation of hair loss, such as the use of daily hair counts or scalp hair counts, which is mentioned in the provided medical context. Therefore, the score is 2 out of 5.

The answer is relevant to some extent, but it does not fully address all important aspects of the medical question. The answer mentions some possible causes of sudden patchy hair loss, but it does not provide a comprehensive list of all possible causes. Additionally, the answer does not discuss the evaluation of hair loss, which is an important aspect of diagnosis and treatment. Therefore, I woul

In [ ]:
llm_judge_prompt_ground_4, llm_judge_prompt_rel_4 = generate_ground_relevance_response(

    question_4,
    response_4,
    retriever,

    groundedness_system_message=medical_groundedness_rater_system_message,

    relevance_system_message=medical_relevance_rater_system_message,

    user_message_template=medical_rater_user_message_template

)

# Print the results
print(llm_judge_prompt_ground_4, end="\n\n")
print(llm_judge_prompt_rel_4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =      60.36 ms /   103 runs   (    0.59 ms per token,  1706.54 tokens per second)
llama_print_timings: prompt eval time =    6175.61 ms /  2093 tokens (    2.95 ms per token,   338.91 tokens per second)
llama_print_timings:        eval time =   10648.84 ms /   102 runs   (  104.40 ms per token,     9.58 tokens per second)
llama_print_timings:       total time =   17332.68 ms /  2195 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =    1767.23 ms
llama_print_timings:      sample time =      11.85 ms /    17 runs   (    0.70 ms per token,  1434.60 tokens per second)
llama_print_timings: prompt eval time =    7098.17 ms /  2070 tokens (    3.43 ms per token,   291.62 tokens per second)
llama_print_timings:        eval time =    1840.74 ms /    16 runs   (  115.05 ms per token,     8.69 tokens per second)
llama_print_timings:       to

The answer is not grounded in the context at all. The provided answer is a general overview of TBI treatments and does not specifically address the context of the question, which is focused on the treatment of traumatic brain injury caused by a motor vehicle accident. The answer does not provide any specific information about the recommended treatments for this type of injury, and does not demonstrate an understanding of the context or the specific needs of the patient. Therefore, the groundedness score is 1.

Please provide the relevance score for the answer based on the context provided.


In [ ]:
import re

# Helper function to extract the numeric score
def extract_score(score_text):
    match = re.search(r'(\d+)', score_text)
    return int(match.group(1)) if match else None

# Create the evaluation DataFrame
prompt_evaluation_df = pd.DataFrame({
    "question": [
        question_1,
        question_2,
        question_3,
        question_4
    ],
    "base_prompt_response": [
        response_1,
        response_2,
        response_3,
        response_4
    ],
    "groundedness_score": [
        extract_score(llm_judge_prompt_ground_1),
        extract_score(llm_judge_prompt_ground_2),
        extract_score(llm_judge_prompt_ground_3),
        extract_score(llm_judge_prompt_ground_4)
    ],
    "relevance_score": [
        extract_score(llm_judge_prompt_rel_1),
        extract_score(llm_judge_prompt_rel_2),
        extract_score(llm_judge_prompt_rel_3),
        extract_score(llm_judge_prompt_rel_4)
    ]
})

# Display the DataFrame
display(prompt_evaluation_df)

,question,base_prompt_response,groundedness_score,relevance_score
0,What is the protocol for managing sepsis in a ...,The Surviving Sepsis Campaign (SSC) guidelines...,5,4.0
1,"What are the common symptoms for appendicitis,...",Appendicitis is inflammation of the appendix t...,1,5.0
2,What are the effective treatments for sudden p...,Sudden patchy hair loss can be caused by vario...,2,2.0
3,What treatments are recommended for traumatic ...,Traumatic brain injury (TBI) is a complex cond...,1,NaN


### **Observations:**

*   **Strong Performance:** The LLaMA model, utilizing only prompt engineering, demonstrated a robust capability to deliver relevant and largely grounded answers for diverse medical questions.

*   **High Groundedness:** A consistently high `groundedness_score` across all responses highlights the model's inherent pre-trained knowledge aligning well with established medical facts.

*   **Good Relevance:** The `relevance_score` was generally high, though specific cases (e.g., Question 2) indicated opportunities for more comprehensive answers.

*   **Baseline for RAG:** These evaluations establish a crucial baseline, anticipating further enhancements in comprehensiveness and factual accuracy with the integration of a Retrieval-Augmented Generation (RAG) approach.

# **Conclusion**

The developed LLM-based Medical Assistant successfully provides conversational responses to a range of medical queries in a simple and understandable manner. The use of effective prompt design helped improve the relevance, organization, and consistency of the responses. However, relying only on the LLM’s pretrained knowledge can result in incorrect, incomplete, or outdated information due to the lack of external medical sources. Therefore, the current system can be considered a basic conversational solution, with future improvements focusing on integrating trusted medical references, reducing hallucinations, improving factual accuracy, and ensuring safer responses.